# 05 — The Real MCP SDK: From DIY to Production

**Goal:** See how the official `mcp` Python SDK implements everything you just built — and how much simpler it is.

## Setup

```bash
pip install mcp
```

## Exercise 5.1 — FastMCP server (the high-level API)

Remember your `mini_mcp_server.py`? Here's the same thing with FastMCP.

Run this cell to create the file.

In [ ]:
server_code = '''\
#!/usr/bin/env python3
"""Same 3 tools as mini_mcp_server.py, but using FastMCP."""
import os
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(name="fastmcp-demo")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
    """
    return a + b

@mcp.tool()
def get_weather(city: str) -> str:
    """Get current weather for a city (fake data for demo).
    
    Args:
        city: City name
    """
    return f"{city}: 28°C, 80% humidity, partly cloudy"

@mcp.tool()
def list_files(path: str = ".") -> str:
    """List files in a directory.
    
    Args:
        path: Directory path (default: current dir)
    """
    files = os.listdir(path)
    return "\\n".join(sorted(files))

if __name__ == "__main__":
    mcp.run()  # stdio transport by default
'''

with open("fastmcp_server.py", "w") as f:
    f.write(server_code)
print("Created fastmcp_server.py")

### Question 5.1 — Compare with your DIY version

Your `mini_mcp_server.py` was ~80 lines. The FastMCP version is ~30 lines.

What did FastMCP handle for you?

| Your DIY version | FastMCP |
|------|------|
| Manual JSON-RPC parsing | ? |
| Manual `initialize` handler | ? |
| Manual `tools/list` with JSON Schema | ? |
| Manual `tools/call` dispatcher | ? |
| Manual stdin/stdout loop | ? |
| Manual error wrapping | ? |

*Fill in what FastMCP does automatically for each row:*


## Exercise 5.2 — Use the official SDK client

Connect to your FastMCP server using the official client SDK.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def test_official_client():
    # Connect to our FastMCP server
    server_params = StdioServerParameters(
        command="python3",
        args=["fastmcp_server.py"],
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize (one line!)
            await session.initialize()
            print(f"Connected to server")
            
            # List tools
            tools_result = await session.list_tools()
            print(f"\nTools ({len(tools_result.tools)}):")
            for t in tools_result.tools:
                print(f"  - {t.name}: {t.description}")
            
            # Call tools
            result = await session.call_tool("add", {"a": 17, "b": 25})
            print(f"\nadd(17, 25) = {result.content[0].text}")
            
            result = await session.call_tool("get_weather", {"city": "Hanoi"})
            print(f"weather = {result.content[0].text}")
            
            result = await session.call_tool("list_files", {"path": "."})
            files = result.content[0].text.split("\n")[:5]
            print(f"files = {files}...")
    
    print("\n✓ Official SDK client works!")

await test_official_client()

## Exercise 5.3 — Connect official client to YOUR DIY server

The beauty of a standard protocol: the official client can talk to your DIY server, and your DIY client can talk to the official server.

In [ ]:
async def test_cross_compatibility():
    """Official SDK client → Your DIY server."""
    server_params = StdioServerParameters(
        command="python3",
        args=["mini_mcp_server.py"],  # YOUR server!
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            tools = await session.list_tools()
            print(f"Official client → DIY server: found {len(tools.tools)} tools")
            for t in tools.tools:
                print(f"  - {t.name}")
            
            result = await session.call_tool("add", {"a": 100, "b": 200})
            print(f"\nadd(100, 200) = {result.content[0].text}")
            assert result.content[0].text == "300"
    
    print("\n✓ Cross-compatibility works! Standard protocol = interoperable.")

await test_cross_compatibility()

### Question 5.3
The official SDK client successfully talked to your hand-built server. What does this tell you about protocols vs implementations?

*Your answer:*


## Exercise 5.4 — Map to sbot Layer 9

Now you understand MCP. Here's how sbot will use it:

```
sbot config:
  mcp_servers:
    - name: filesystem
      command: ["npx", "-y", "@modelcontextprotocol/server-filesystem", "/path"]
    - name: my-tools  
      command: ["python3", "my_mcp_server.py"]

sbot startup:
  1. For each MCP server config:
     a. Launch subprocess (MiniMCPClient.start)
     b. Initialize handshake (MiniMCPClient.initialize)
     c. Discover tools (MiniMCPClient.list_tools)
     d. Create LangChain @tool wrappers for each MCP tool
  2. Merge MCP tools into TOOLS list
  3. llm.bind_tools(TOOLS)  ← now includes MCP tools

sbot runtime:
  LLM says: call "filesystem__read_file" with {"path": "/etc/hosts"}
  → sbot finds this is an MCP tool from the "filesystem" server
  → forwards to MiniMCPClient.call_tool("read_file", {"path": "/etc/hosts"})
  → returns result to LLM as ToolMessage
```

### Question 5.4
In the sbot flow above, why does the tool name become `filesystem__read_file` (with a prefix) instead of just `read_file`?

**Hint:** What happens if two MCP servers both expose a tool called `read_file`?

*Your answer:*


## What You Built

```
Notebook 01: JSON-RPC 2.0 message format + dispatcher
Notebook 02: stdio transport (newline-delimited JSON over pipes)
Notebook 03: MCP server (initialize + tools/list + tools/call)
Notebook 04: MCP client (subprocess + handshake + tool calling)
Notebook 05: Official SDK (FastMCP) + cross-compatibility proof
```

## Key Takeaways

1. **MCP = JSON-RPC 2.0 + a few standard methods** (initialize, tools/list, tools/call)
2. **stdio transport is trivially simple** — one JSON per line over stdin/stdout
3. **The handshake exchanges capabilities** — client and server agree on what they support
4. **Tool errors ≠ protocol errors** — tool failures are successful responses with `isError: true`
5. **Standard protocol = interoperability** — your DIY server works with the official client
6. **FastMCP abstracts the boilerplate** — you write Python functions, it handles everything else

## What's NOT covered (but follows the same pattern)

- **Resources**: `resources/list` + `resources/read` (like tools, but read-only data)
- **Prompts**: `prompts/list` + `prompts/get` (reusable prompt templates)
- **Sampling**: server asks client for LLM completions (reverse direction)
- **Streamable HTTP transport**: HTTP POST instead of stdio (for remote servers)

All of these follow the same JSON-RPC request/response pattern you've already mastered.